# Module 4.1: K-Means & DBSCAN for Machine Operational Regime Detection
### 🇵🇱 Algorytmy K-Średnich i DBSCAN w Wykrywaniu Reżimów Pracy Maszyn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m04_l01_kmeans_dbscan_regimes.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Discover multi-state CNC operating modes (Idle, Warmup, Full Speed) without human labels using K-Means and isolate anomalous points with DBSCAN.  
> 🇵🇱 **Streszczenie:** Odkrywaj stany pracy obrabiarki (Bieg jałowy, Rozgrzewanie, Pełna moc) bez etykiet za pomocą K-Średnich i izoluj anomalie z DBSCAN.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Operational Machine Regimes**: CNC mills switch between Standby, Warmup, Rough Milling, and High-Speed Finishing. Thresholds set for finishing cause false alarms during roughing.
2. **K-Means Centroids**: Act as golden reference signatures for healthy operating modes.
3. **DBSCAN for Non-Convex Envelopes**: Density-Based Spatial Clustering identifies arbitrary boundary shapes and automatically labels unmodeled operational deviations as noise (`label = -1`).


In [ ]:
# 1. Simulate Multi-Mode CNC Spindle Regimes (RPM vs Motor Current)
from sklearn.cluster import KMeans, DBSCAN

np.random.seed(42)
# Regime 1: Standby (RPM: ~300, Current: ~5A)
c1 = np.random.multivariate_normal([300, 5], [[400, 5], [5, 0.8]], size=180)
# Regime 2: Normal Roughing (RPM: ~2400, Current: ~35A)
c2 = np.random.multivariate_normal([2400, 35], [[3000, 25], [25, 4.0]], size=250)
# Regime 3: High-Speed Finishing (RPM: ~4800, Current: ~22A)
c3 = np.random.multivariate_normal([4800, 22], [[4000, 20], [20, 3.0]], size=180)

# Unmodeled breakdown deviations (tool binding / mechanical seizure)
anomalies = np.array([[1200, 55], [1600, 68], [3400, 75], [500, 38], [4200, 62]])

X_regimes = np.vstack([c1, c2, c3, anomalies])
print(f"✓ Total operational data points: {len(X_regimes)}")


In [ ]:
# 2. Fit K-Means Regimes and DBSCAN Anomaly Envelopes
kmeans = KMeans(n_clusters=3, random_state=42).fit(X_regimes)
dbscan = DBSCAN(eps=180.0, min_samples=8).fit(X_regimes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means Centroid Clusters
axes[0].scatter(X_regimes[:, 0], X_regimes[:, 1], c=kmeans.labels_, cmap='viridis', alpha=0.7)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c='red', marker='X', s=200, label='Centroids')
axes[0].set_title("K-Means: Regime Segmentation", fontweight='bold')
axes[0].set_xlabel("Spindle RPM")
axes[0].set_ylabel("Motor Current [Amps]")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# DBSCAN Anomaly Flags (label == -1)
noise_mask = dbscan.labels_ == -1
axes[1].scatter(X_regimes[~noise_mask, 0], X_regimes[~noise_mask, 1], c=dbscan.labels_[~noise_mask], cmap='plasma', alpha=0.6, label='Operational Envelopes')
axes[1].scatter(X_regimes[noise_mask, 0], X_regimes[noise_mask, 1], color='#ef4444', s=120, edgecolors='black', label='DBSCAN Flagged Anomalies')
axes[1].set_title("DBSCAN: Density Anomaly Isolation", fontweight='bold')
axes[1].set_xlabel("Spindle RPM")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Partition multi-sensor CNC machine logs using K-Means across clusters K=2 to K=6 *(🇵🇱 Podziel logi z maszyny CNC algorytmem K-Means dla liczby klastrów od 2 do 6)*
- [ ] **Task 2:** Compute Elbow curve inertia and Silhouette scores to select the optimal regime count *(🇵🇱 Wykreśl krzywą inercji (metoda łokcia) i wskaźnik Silhouette, wybierając optymalną liczbę reżimów)*
- [ ] **Task 3:** Deploy DBSCAN with tuned epsilon (eps) to detect non-spherical clusters and flag noise as anomalies *(🇵🇱 Zastosuj DBSCAN z dobranym parametrem epsilon do detekcji nieregularnych klastrów i izolacji szumu)*


In [ ]:
# Exercise Challenge:
# Assert that DBSCAN flags all 5 injected physical anomalies as noise
flagged_count = (dbscan.labels_ == -1).sum()
print(f"DBSCAN Isolated Outliers: {flagged_count} (Expected >= 5)")
assert flagged_count >= 5, "DBSCAN failed to isolate operational anomalies."
print("✓ Operational regime clustering test passed.")
